# ChemBERTa Embedding Generation

This notebook generates molecular embedding vectors for each drug using the pre-trained ChemBERTa model.

**Input:** `drugbank_smiles.csv` (output of the SMILES extraction notebook)  
**Output:** `chembert_embeddings.csv` — one row per drug, 768 embedding dimensions

**Run order:**
1. `drugbank_smiles_extraction.ipynb` ✅
2. `chembert_embeddings.ipynb` ← you are here
3. `knn_baseline.ipynb`

**Note:** This notebook may take 10–30 minutes depending on your machine. It runs on CPU by default.

## Install Dependencies

Run this cell once if you haven't installed these packages yet.

In [1]:
import sys
!{sys.executable} -m pip install transformers torch pandas --quiet
print('Dependencies ready')

Dependencies ready


## Imports

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

PyTorch version: 2.11.0
Device: CPU


## Step 1 — Load SMILES Data

In [ ]:
smiles_df = pd.read_csv('data/drugbank_smiles.csv')

print(f'Drugs to embed: {len(smiles_df):,}')
smiles_df.head()

Drugs to embed: 14,627


,drugbank_id,smiles
0,DB00006,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...
1,DB00014,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,DB00027,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,DB00035,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...
4,DB00050,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...


## Step 2 — Load ChemBERTa Model

Downloads the model on first run (~400MB). Cached locally after that.

In [4]:
MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print('Model loaded successfully')

Loading seyonec/ChemBERTa-zinc-base-v1...


config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


## Step 3 — Generate Embeddings

Each SMILES string is passed through ChemBERTa. We use **mean pooling** over the token embeddings to produce a single 768-dimensional vector per drug.

Progress is printed every 500 drugs.

In [5]:
embeddings = {}
failed     = []

for i, row in smiles_df.iterrows():
    drug_id = row['drugbank_id']
    smiles  = row['smiles']

    try:
        inputs = tokenizer(
            smiles,
            return_tensors='pt',
            max_length=512,
            truncation=True,
            padding=True
        )
        with torch.no_grad():
            outputs = model(**inputs)

        # Mean pool over token dimension -> single 768-dim vector
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embeddings[drug_id] = embedding

    except Exception as e:
        failed.append(drug_id)

    if (i + 1) % 500 == 0:
        print(f'  Processed {i+1:,} / {len(smiles_df):,} drugs...')

print(f'\nSuccessfully embedded: {len(embeddings):,} drugs')
print(f'Failed:                {len(failed):,} drugs')

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

  Processed 500 / 14,627 drugs...
  Processed 1,000 / 14,627 drugs...
  Processed 1,500 / 14,627 drugs...
  Processed 2,000 / 14,627 drugs...
  Processed 2,500 / 14,627 drugs...
  Processed 3,000 / 14,627 drugs...
  Processed 3,500 / 14,627 drugs...
  Processed 4,000 / 14,627 drugs...
  Processed 4,500 / 14,627 drugs...
  Processed 5,000 / 14,627 drugs...
  Processed 5,500 / 14,627 drugs...
  Processed 6,000 / 14,627 drugs...
  Processed 6,500 / 14,627 drugs...
  Processed 7,000 / 14,627 drugs...
  Processed 7,500 / 14,627 drugs...
  Processed 8,000 / 14,627 drugs...
  Processed 8,500 / 14,627 drugs...
  Processed 9,000 / 14,627 drugs...
  Processed 9,500 / 14,627 drugs...
  Processed 10,000 / 14,627 drugs...
  Processed 10,500 / 14,627 drugs...
  Processed 11,000 / 14,627 drugs...
  Processed 11,500 / 14,627 drugs...
  Processed 12,000 / 14,627 drugs...
  Processed 12,500 / 14,627 drugs...
  Processed 13,000 / 14,627 drugs...
  Processed 13,500 / 14,627 drugs...
  Processed 14,000 / 1

## Step 4 — Save Embeddings to CSV

In [ ]:
emb_df = pd.DataFrame(embeddings).T
emb_df.index.name = 'drugbank_id'

print(f'Embedding matrix shape: {emb_df.shape}')
print(f'  Rows = drugs, Columns = embedding dimensions')

emb_df.to_csv('data/chembert_embeddings.csv')
print('\nSaved to chembert_embeddings.csv')
print('You can now run knn_baseline.ipynb')

Embedding matrix shape: (14627, 768)
  Rows = drugs, Columns = embedding dimensions

Saved to chembert_embeddings.csv
You can now run knn_baseline.ipynb


## Step 5 — Quick Sanity Check

In [ ]:
# Reload and verify the saved file looks correct
check = pd.read_csv('data/chembert_embeddings.csv', index_col='drugbank_id')

print(f'Drugs loaded:          {len(check):,}')
print(f'Embedding dimensions:  {check.shape[1]}')
print(f'Any NaN values:        {check.isna().any().any()}')
print(f'\nSample (first 3 drugs, first 5 dims):')
check.iloc[:3, :5]

Drugs loaded:          14,627
Embedding dimensions:  768
Any NaN values:        False

Sample (first 3 drugs, first 5 dims):


,0,1,2,3,4
drugbank_id,,,,,
DB00006,0.123676,-0.116291,-0.324462,-0.186317,-0.018497
DB00014,0.273169,0.120439,-0.304858,-0.496637,0.184836
DB00027,0.442134,0.070210,-0.217950,-0.457789,0.273419
